In [ ]:
# In[1]:

import pandas as pd

# Load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes as required (keep raw unix seconds for min/max)
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['ts_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['ts_utc'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Helper to build summary and top-n for name-like column
def summarize_name_counts(df, name_col, file_label):
    total_rows = len(df)
    cmdb_ids = sorted(df['cmdb_id'].dropna().astype(str).unique())
    min_ts = int(df['timestamp'].min()) if total_rows>0 else None
    max_ts = int(df['timestamp'].max()) if total_rows>0 else None
    summary = pd.DataFrame([{
        'file': file_label,
        'total_rows': total_rows,
        'cmdb_ids': ",".join(cmdb_ids),
        'min_timestamp': min_ts,
        'max_timestamp': max_ts
    }])
    # top up to 20 distinct names with counts
    if name_col in df.columns:
        top = df[name_col].fillna("<NA>").astype(str).value_counts().reset_index()
        top.columns = [name_col, 'count']
        top = top.head(20)
    else:
        top = pd.DataFrame(columns=[name_col, 'count'])
    return summary, top

metric_summary, metric_top_kpis = summarize_name_counts(df_metric, 'kpi_name', 'metric.csv')
trace_summary, trace_top = summarize_name_counts(df_trace, 'trace_name', 'trace.csv')
log_summary, log_top = summarize_name_counts(df_log, 'log_name', 'log.csv')

# For error_logs, use first 100 chars of message
df_error['message_short'] = df_error['message'].astype(str).str.slice(0,100)
error_summary = pd.DataFrame([{
    'file':'error_logs.csv',
    'total_rows': len(df_error),
    'cmdb_ids': ",".join(sorted(df_error['cmdb_id'].dropna().astype(str).unique())),
    'min_timestamp': int(df_error['timestamp'].min()) if len(df_error)>0 else None,
    'max_timestamp': int(df_error['timestamp'].max()) if len(df_error)>0 else None
}])
error_top = df_error['message_short'].value_counts().reset_index()
error_top.columns = ['message_short_100', 'count']
error_top = error_top.head(20)

# Display compact aggregates
metric_summary, metric_top_kpis, trace_summary, trace_top, log_summary, log_top, error_summary, error_top

```
Out[1]:
```
```python
# Summarize the previously computed aggregates in plain English as a single string.
# Reuse existing variables from prior execution: metric_summary, metric_top_kpis, trace_summary, trace_top, log_summary, log_top, error_summary, error_top

def top_pairs_str(df, name_col, max_items=20):
    if df is None or df.shape[0]==0:
        return "(none)"
    pairs = [f"{row[0]}({int(row[1])})" for row in df.values[:max_items]]
    return ", ".join(pairs)

# Extract summary values (safe access)
m_sum = metric_summary.loc[0] if 'metric_summary' in globals() else None
t_sum = trace_summary.loc[0] if 'trace_summary' in globals() else None
l_sum = log_summary.loc[0] if 'log_summary' in globals() else None
e_sum = error_summary.loc[0] if 'error_summary' in globals() else None

metric_total = int(m_sum['total_rows']) if m_sum is not None else 0
metric_cmdbs = m_sum['cmdb_ids'] if (m_sum is not None) else ""
metric_min_ts = int(m_sum['min_timestamp']) if (m_sum is not None and pd.notna(m_sum['min_timestamp'])) else None
metric_max_ts = int(m_sum['max_timestamp']) if (m_sum is not None and pd.notna(m_sum['max_timestamp'])) else None
metric_top_str = top_pairs_str(metric_top_kpis, 'kpi_name')

trace_total = int(t_sum['total_rows']) if t_sum is not None else 0
trace_cmdbs = t_sum['cmdb_ids'] if (t_sum is not None) else ""
trace_min_ts = int(t_sum['min_timestamp']) if (t_sum is not None and pd.notna(t_sum['min_timestamp'])) else None
trace_max_ts = int(t_sum['max_timestamp']) if (t_sum is not None and pd.notna(t_sum['max_timestamp'])) else None
trace_top_str = top_pairs_str(trace_top, 'trace_name')

log_total = int(l_sum['total_rows']) if l_sum is not None else 0
log_cmdbs = l_sum['cmdb_ids'] if (l_sum is not None) else ""
log_min_ts = int(l_sum['min_timestamp']) if (l_sum is not None and pd.notna(l_sum['min_timestamp'])) else None
log_max_ts = int(l_sum['max_timestamp']) if (l_sum is not None and pd.notna(l_sum['max_timestamp'])) else None
log_top_str = top_pairs_str(log_top, 'log_name')

error_total = int(e_sum['total_rows']) if e_sum is not None else 0
error_cmdbs = e_sum['cmdb_ids'] if (e_sum is not None) else ""
error_min_ts = None if e_sum is None else (int(e_sum['min_timestamp']) if pd.notna(e_sum['min_timestamp']) else None)
error_max_ts = None if e_sum is None else (int(e_sum['max_timestamp']) if pd.notna(e_sum['max_timestamp']) else None)
error_top_str = top_pairs_str(error_top, 'message_short_100')

summary = (
    "Aggregate summary (plain English):\n\n"
    f"1) metric.csv\n"
    f"   - Total rows: {metric_total}\n"
    f"   - Distinct cmdb_id values: {metric_cmdbs}\n"
    f"   - Top KPI names (name(count), up to 20): {metric_top_str}\n"
    f"   - Min timestamp (unix seconds): {metric_min_ts}, Max timestamp: {metric_max_ts}\n\n"
    f"2) trace.csv\n"
    f"   - Total rows: {trace_total}\n"
    f"   - Distinct cmdb_id values: {trace_cmdbs}\n"
    f"   - Top trace names (name(count), up to 20): {trace_top_str}\n"
    f"   - Min timestamp (unix seconds): {trace_min_ts}, Max timestamp: {trace_max_ts}\n\n"
    f"3) log.csv\n"
    f"   - Total rows: {log_total}\n"
    f"   - Distinct cmdb_id values: {log_cmdbs}\n"
    f"   - Top log names (name(count), up to 20): {log_top_str}\n"
    f"   - Min timestamp (unix seconds): {log_min_ts}, Max timestamp: {log_max_ts}\n\n"
    f"4) error_logs.csv\n"
    f"   - Total rows: {error_total}\n"
    f"   - Distinct cmdb_id values: {error_cmdbs or '(none)'}\n"
    f"   - Top messages (first 100 chars with counts): {error_top_str}\n"
    f"   - Min timestamp (unix seconds): {error_min_ts}, Max timestamp: {error_max_ts}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows                                           cmdb_ids  min_timestamp  max_timestamp
0  metric.csv        8977  ts-admin-basic-info-service,ts-admin-order-ser...     1706151060     1706152500,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    700
5  latency-50    699
6  latency-90    699
7       error    104,         file  total_rows                                           cmdb_ids  min_timestamp  max_timestamp
0  trace.csv       18840  root,ts-admin-basic-info-service,ts-admin-trav...     1706151060     1706152500,                                            trace_name  count
0                           trace.from_root.row_count    350
1                          trace.from_root.error_rate    350
2                        trace.from_root.duration_p95    350
3                       trace.from_root.duration_mean    350
4        trace.from_ts-preserve-service.duration_mean    268
5         trace.from_ts-preserve-service.duration_p95    268
6           trace.from_ts-preserve-service.error_rate    268
7            trace.from_ts-preserve-service.row_count    268
8      trace.from_ts-preserve-other-service.row_count    244
9     trace.from_ts-preserve-other-service.error_rate    244
10  trace.from_ts-preserve-other-service.duration_p95    244
11  trace.from_ts-preserve-other-service.duration_...    244
12            trace.to_ts-order-service.duration_mean    175
13             trace.to_ts-order-service.duration_p95    175
14               trace.to_ts-order-service.error_rate    175
15                trace.to_ts-order-service.row_count    175
16              trace.to_ts-station-service.row_count    173
17             trace.to_ts-station-service.error_rate    173
18           trace.to_ts-station-service.duration_p95    173
19          trace.to_ts-station-service.duration_mean    173,       file  total_rows                                           cmdb_ids  min_timestamp  max_timestamp
0  log.csv        2198  ts-admin-basic-info-service,ts-admin-travel-se...     1706151060     1706152500,           log_name  count
0  log.error_count   1099
1    log.row_count   1099,              file  total_rows cmdb_ids min_timestamp max_timestamp
0  error_logs.csv           0                   None          None, Empty DataFrame
Columns: [message_short_100, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Ensure df_metric is available (reuse if present, otherwise load)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Parse timestamp to UTC datetime column (keep raw unix seconds in 'timestamp')
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure 'value' is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Filter to requested cmdb_id set
target_cmdbs = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
df_filtered = df_metric[df_metric['cmdb_id'].isin(target_cmdbs)].copy()

# Aggregate per (cmdb_id, kpi_name) using the full available series
agg = df_filtered.groupby(['cmdb_id', 'kpi_name']).agg(
    count = ('value', 'count'),
    min_timestamp = ('timestamp', 'min'),
    max_timestamp = ('timestamp', 'max'),
    p50 = ('value', lambda s: s.quantile(0.50)),
    p90 = ('value', lambda s: s.quantile(0.90)),
    p95 = ('value', lambda s: s.quantile(0.95))
).reset_index()

# Ensure timestamp columns are ints
agg['min_timestamp'] = agg['min_timestamp'].astype('Int64')
agg['max_timestamp'] = agg['max_timestamp'].astype('Int64')

# Sort by cmdb_id then kpi_name
metric_agg_global = agg.sort_values(['cmdb_id', 'kpi_name']).reset_index(drop=True)

# Display a compact result (limit to at most 50 rows)
metric_agg_global.head(50)

```
Out[2]:
```
```python
# Build a plain-English summary from the previously computed aggregation `metric_agg_global`.
# Reuse the kernel variable `metric_agg_global` created earlier.

# Safety check
try:
    metric_agg_global
except NameError:
    raise RuntimeError("metric_agg_global is not available in the kernel. Run the aggregation step first.")

# Basic facts
n_rows = int(metric_agg_global.shape[0])
cmdb_list = sorted(metric_agg_global['cmdb_id'].unique().tolist())
count_min = int(metric_agg_global['count'].min())
count_max = int(metric_agg_global['count'].max())
ts_min = int(metric_agg_global['min_timestamp'].min())
ts_max = int(metric_agg_global['max_timestamp'].max())

# Top 3 p95 overall
top3_overall = metric_agg_global.sort_values('p95', ascending=False).head(3)[['cmdb_id','kpi_name','p50','p90','p95']]

# Top 3 p95 among latency KPIs (if any)
latency_mask = metric_agg_global['kpi_name'].str.contains('latency', na=False)
top3_latency = metric_agg_global[latency_mask].sort_values('p95', ascending=False).head(3)[['cmdb_id','kpi_name','p50','p90','p95']]

# Helper to format numeric values compactly
def fmt(x):
    if pd.isna(x):
        return "NA"
    return f"{float(x):.6g}"

# Format top lists into short lines
def format_rows(df):
    if df.shape[0] == 0:
        return "(none)"
    lines = []
    for _, r in df.iterrows():
        lines.append(f"{r['cmdb_id']} / {r['kpi_name']}: p95={fmt(r['p95'])}, p90={fmt(r['p90'])}, p50={fmt(r['p50'])}")
    return "; ".join(lines)

import pandas as pd  # ensure available for formatting checks

summary = (
    f"Summary of metric.csv aggregation for the 5 target services:\n\n"
    f"- Total (cmdb_id, kpi_name) rows produced: {n_rows}.\n"
    f"- Distinct cmdb_id values included: {', '.join(cmdb_list)}.\n"
    f"- Count of points per combination ranges from {count_min} to {count_max} (most combos have 25 points).\n"
    f"- Time span (raw unix seconds): min={ts_min}, max={ts_max}.\n\n"
    f"- Top 3 highest p95 values overall:\n  {format_rows(top3_overall)}\n\n"
    f"- Top 3 highest p95 values among latency KPIs:\n  {format_rows(top3_latency)}\n\n"
    f"The full aggregated table is stored in the kernel variable `metric_agg_global` "
    f"and contains the columns: cmdb_id, kpi_name, count, min_timestamp, max_timestamp, p50, p90, p95."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count  min_timestamp  max_timestamp           p50           p90           p95
0     ts-auth-service         cpu     25     1706151060     1706152500  1.351204e+01  1.571343e+01  1.621919e+01
1     ts-auth-service      diskio     25     1706151060     1706152500  5.083446e+04  5.901450e+04  6.011815e+04
2     ts-auth-service  latency-50     25     1706151060     1706152500  2.319874e-01  2.504683e-01  2.526689e-01
3     ts-auth-service  latency-90     25     1706151060     1706152500  6.570234e-01  8.774491e-01  9.549557e-01
4     ts-auth-service         mem     25     1706151060     1706152500  2.466891e+08  2.471773e+08  2.472484e+08
..                ...         ...    ...            ...            ...           ...           ...           ...
32  ts-travel-service  latency-50     25     1706151060     1706152500  1.764702e-02  5.958998e-01  6.007788e-01
33  ts-travel-service  latency-90     25     1706151060     1706152500  2.802570e-01  4.966914e+00  5.232921e+00
34  ts-travel-service         mem     25     1706151060     1706152500  2.876035e+08  2.903027e+08  2.915942e+08
35  ts-travel-service      socket     25     1706151060     1706152500  2.068333e+01  3.560000e+01  3.589333e+01
36  ts-travel-service    workload     25     1706151060     1706152500  8.160033e+00  9.588515e+00  9.838854e+00

[37 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing df_metric and metric_agg_global if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    metric_agg_global
except NameError:
    raise RuntimeError("metric_agg_global not found in kernel. Run the previous aggregation step first.")

# Ensure correct dtypes and helper columns
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Target cmdb_ids and incident window (UTC)
target_cmdbs = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
start_ts = int(pd.Timestamp("2024-01-25 02:48:38", tz="UTC").timestamp())
end_ts   = int(pd.Timestamp("2024-01-25 03:18:38", tz="UTC").timestamp())

# Filter metric rows for targets and window; store in kernel variable metric_window_df
metric_window_df = df_metric[
    (df_metric['cmdb_id'].isin(target_cmdbs)) &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# p95 thresholds from metric_agg_global
p95_df = metric_agg_global[['cmdb_id','kpi_name','p95']].drop_duplicates().copy()

# Join window rows with p95 only for combos that exist in metric_agg_global
win_with_p95 = metric_window_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='inner')

# Group by and compute stats for combos that have rows in the window
def compute_stats(g):
    cnt = int(g['value'].count())
    n_anom = int((g['value'] >= g['p95']).sum())
    earliest = int(g.loc[g['value'] >= g['p95'], 'timestamp'].min()) if n_anom > 0 else pd.NA
    max_v = float(g['value'].max()) if cnt > 0 else pd.NA
    mean_v = float(g['value'].mean()) if cnt > 0 else pd.NA
    return pd.Series({
        'count_in_window': cnt,
        'n_anomaly_ge_p95': n_anom,
        'earliest_anom_ts': earliest,
        'max_value_in_window': max_v,
        'mean_value_in_window': mean_v
    })

if not win_with_p95.empty:
    agg_win = win_with_p95.groupby(['cmdb_id','kpi_name']).apply(compute_stats).reset_index()
else:
    # empty dataframe with expected columns
    agg_win = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','n_anomaly_ge_p95',
                                    'earliest_anom_ts','max_value_in_window','mean_value_in_window'])

# Ensure every (cmdb_id,kpi_name) present in metric_agg_global is represented
all_combos = metric_agg_global[['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)
metric_anomaly_summary = all_combos.merge(agg_win, on=['cmdb_id','kpi_name'], how='left')

# Fill missing numeric counts with 0 and keep earliest_anom_ts as NA where absent
metric_anomaly_summary['count_in_window'] = metric_anomaly_summary['count_in_window'].fillna(0).astype(int)
metric_anomaly_summary['n_anomaly_ge_p95'] = metric_anomaly_summary['n_anomaly_ge_p95'].fillna(0).astype(int)
metric_anomaly_summary['earliest_anom_ts'] = metric_anomaly_summary['earliest_anom_ts'].where(
    pd.notna(metric_anomaly_summary['earliest_anom_ts']), pd.NA
)
metric_anomaly_summary['max_value_in_window'] = metric_anomaly_summary['max_value_in_window'].astype('Float64')
metric_anomaly_summary['mean_value_in_window'] = metric_anomaly_summary['mean_value_in_window'].astype('Float64')

# Sort by n_anomaly_ge_p95 desc then max_value_in_window desc
metric_anomaly_summary = metric_anomaly_summary.sort_values(
    by=['n_anomaly_ge_p95','max_value_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

# Prepare compact display (top 50 rows)
metric_anomaly_summary_display = metric_anomaly_summary.head(50)

# Expose metric_window_df and metric_anomaly_summary in the kernel as requested
metric_anomaly_summary_display, metric_window_df.shape

```
Out[3]:
```
```python
# Summarize the anomaly analysis in plain English using existing kernel variables:
# metric_anomaly_summary (DataFrame) and metric_window_df (DataFrame)

import pandas as pd

# Safety checks
if 'metric_anomaly_summary' not in globals():
    raise RuntimeError("metric_anomaly_summary not found in kernel.")
if 'metric_window_df' not in globals():
    raise RuntimeError("metric_window_df not found in kernel.")

df = metric_anomaly_summary.copy()
win_shape = metric_window_df.shape

# Count combos with any anomalies (n_anomaly_ge_p95 > 0)
num_anom_combos = int((df['n_anomaly_ge_p95'] > 0).sum())
total_combos = int(df.shape[0])
total_window_rows = int(win_shape[0])

# Prepare top rows (limit 10) for concise display
top_rows = df.sort_values(['n_anomaly_ge_p95','max_value_in_window'], ascending=[False, False]).head(10)

def fmt_num(x):
    if pd.isna(x):
        return "None"
    return f"{float(x):.6g}"

def fmt_ts(x):
    if pd.isna(x):
        return "None"
    return str(int(x))

lines = []
for _, r in top_rows.iterrows():
    lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} : count_in_window={int(r['count_in_window'])}, "
        f"n_anomaly_ge_p95={int(r['n_anomaly_ge_p95'])}, earliest_anom_ts={fmt_ts(r['earliest_anom_ts'])}, "
        f"max={fmt_num(r['max_value_in_window'])}, mean={fmt_num(r['mean_value_in_window'])}"
    )

summary = (
    f"Incident-window metrics summary (2024-01-25 02:48:38 to 2024-01-25 03:18:38 UTC):\n\n"
    f"- Total metric rows in window for the 5 services: {total_window_rows}.\n"
    f"- Total (cmdb_id, kpi_name) combinations considered: {total_combos}.\n"
    f"- Combinations with at least one value >= historical p95: {num_anom_combos}.\n\n"
    f"- Top 10 combinations by anomaly count and peak value:\n  " + "\n  ".join(lines) + "\n\n"
    f"- Notable observation: memory ('mem') metrics for each service show anomalies (n_anomaly_ge_p95=2) "
    f"with elevated max/mean values in the window. Other KPI types (latency, error, socket, etc.) also "
    f"appear in the summary results above.\n\n"
    f"The full per-combination summary table is saved in kernel variable `metric_anomaly_summary` "
    f"and the filtered window rows are in `metric_window_df`."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count_in_window  n_anomaly_ge_p95  earliest_anom_ts  max_value_in_window  mean_value_in_window
0   ts-travel-service         mem               25                 2      1.706152e+09     293271184.410256      288693811.204774
1    ts-order-service         mem               25                 2      1.706152e+09     265977225.846154      265279631.846937
2    ts-route-service         mem               25                 2      1.706152e+09     257225124.102564       253810466.98786
3    ts-train-service         mem               25                 2      1.706152e+09          251274240.0      250699275.638751
4     ts-auth-service         mem               25                 2      1.706152e+09     247275099.897436      246745180.422564
..                ...         ...              ...               ...               ...                  ...                   ...
32   ts-order-service  latency-50               25                 2      1.706152e+09             0.013488              0.007468
33   ts-route-service  latency-50               25                 2      1.706152e+09             0.004782              0.003899
34   ts-train-service  latency-50               25                 2      1.706152e+09             0.004623              0.003973
35   ts-route-service       error                2                 2      1.706152e+09                  0.0                   0.0
36  ts-travel-service       error                2                 2      1.706152e+09                  0.0                   0.0

[37 rows x 7 columns], (879, 5))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure correct dtypes and add UTC datetime column (timestamps are unix seconds)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Global aggregation across the full trace file per trace_name
trace_agg_global = (
    df_trace.groupby('trace_name', dropna=False)['value']
    .agg(
        count = 'count',
        p50 = lambda s: s.quantile(0.50),
        p90 = lambda s: s.quantile(0.90),
        p95 = lambda s: s.quantile(0.95)
    )
    .reset_index()
)

# Add min/max timestamps (raw unix seconds) per trace_name using full series
ts_bounds = (
    df_trace.groupby('trace_name', dropna=False)['timestamp']
    .agg(min_timestamp='min', max_timestamp='max')
    .reset_index()
)

trace_agg_global = trace_agg_global.merge(ts_bounds, on='trace_name', how='left')

# Ensure types and column order
trace_agg_global['min_timestamp'] = trace_agg_global['min_timestamp'].astype('Int64')
trace_agg_global['max_timestamp'] = trace_agg_global['max_timestamp'].astype('Int64')
trace_agg_global = trace_agg_global[['trace_name','count','min_timestamp','max_timestamp','p50','p90','p95']]

# Filter to trace_name containing any of the target substrings (case-sensitive)
substrs = [
    'to_ts-auth-service','to_ts-order-service','to_ts-route-service','to_ts-train-service','to_ts-travel-service',
    'from_ts-auth-service','from_ts-order-service','from_ts-route-service','from_ts-train-service','from_ts-travel-service'
]
pattern = "|".join(substrs)
trace_agg_filtered = trace_agg_global[trace_agg_global['trace_name'].str.contains(pattern, na=False)]

# Sort by p95 descending and limit to top 20 rows for compact display
trace_agg_filtered_top20 = trace_agg_filtered.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

# Display the compact table (top 20) and keep full aggregation in trace_agg_global
trace_agg_filtered_top20, trace_agg_global.shape

```
Out[4]:
```
```python
# Summarize the filtered trace aggregation in plain English as a single string.
# Reuse existing kernel variables: trace_agg_filtered_top20, trace_agg_global

import pandas as pd

# Safety checks
if 'trace_agg_filtered_top20' not in globals() or 'trace_agg_global' not in globals():
    raise RuntimeError("Required variables not found in kernel: trace_agg_filtered_top20 and/or trace_agg_global")

df = trace_agg_filtered_top20.copy()
total_filtered = int(df.shape[0])
total_global = int(trace_agg_global.shape[0])
min_ts = int(df['min_timestamp'].min())
max_ts = int(df['max_timestamp'].max())

# Top 5 by p95
top5 = df.sort_values('p95', ascending=False).head(5)
top5_lines = []
for _, r in top5.iterrows():
    top5_lines.append(f"{r['trace_name']} (count={int(r['count'])}, p95={float(r['p95']):.6g})")

summary = (
    f"Trace aggregation summary (filtered to traces to/from the 5 ts-* services):\n\n"
    f"- Filtered groups returned: {total_filtered} (out of {total_global} total trace_name groups in trace_agg_global).\n"
    f"- Time span across these groups (raw unix seconds): min={min_ts}, max={max_ts}.\n\n"
    f"- Top 5 trace_name entries by p95:\n  " + "\n  ".join(top5_lines) + "\n\n"
    f"- Notable observations: The highest p95 values are for row_count traces (thousands of rows at p95) "
    f"for train and route services, e.g. trace.from_ts-train-service.row_count p95={float(top5.iloc[0]['p95']):.6g}. "
    f"Also, duration p95 for trace.to_ts-travel-service is relatively large (p95={float(df.loc[df['trace_name']=='trace.to_ts-travel-service.duration_p95','p95'].squeeze()):.6g}).\n\n"
    f"The full global aggregation is saved in kernel variable `trace_agg_global`."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  count  min_timestamp  max_timestamp          p50          p90          p95
0        trace.from_ts-train-service.row_count     25     1706151060     1706152500  2224.000000  2702.400000  2833.600000
1          trace.to_ts-train-service.row_count    100     1706151060     1706152500   218.000000  2410.000000  2640.400000
2        trace.from_ts-route-service.row_count     25     1706151060     1706152500  1738.000000  2175.200000  2262.400000
3          trace.to_ts-route-service.row_count    100     1706151060     1706152500   316.500000  1967.800000  2172.100000
4         trace.to_ts-travel-service.row_count    150     1706151060     1706152500    19.000000  1313.600000  1440.300000
5       trace.from_ts-travel-service.row_count    150     1706151060     1706152500   210.000000  1313.600000  1440.300000
6        trace.from_ts-order-service.row_count     50     1706151060     1706152500    57.500000   451.300000   492.550000
7          trace.to_ts-order-service.row_count    175     1706151060     1706152500     9.000000   372.600000   422.300000
8         trace.from_ts-auth-service.row_count     25     1706151060     1706152500   286.000000   354.400000   359.200000
9           trace.to_ts-auth-service.row_count     50     1706151060     1706152500   109.000000   348.200000   354.200000
10     trace.to_ts-travel-service.duration_p95    150     1706151060     1706152500     0.422814     6.090193    16.023874
11    trace.to_ts-travel-service.duration_mean    150     1706151060     1706152500     0.307792     5.794865    11.040090
12   trace.from_ts-travel-service.duration_p95    150     1706151060     1706152500     0.030518     1.444177     1.449913
13  trace.from_ts-travel-service.duration_mean    150     1706151060     1706152500     0.008063     0.303578     1.284184
14       trace.to_ts-auth-service.duration_p95     50     1706151060     1706152500     0.168212     0.850990     1.103418
15    trace.from_ts-order-service.duration_p95     50     1706151060     1706152500     0.199749     0.519917     0.526539
16      trace.to_ts-order-service.duration_p95    175     1706151060     1706152500     0.010803     0.474809     0.526069
17   trace.from_ts-order-service.duration_mean     50     1706151060     1706152500     0.189670     0.498260     0.516655
18     trace.to_ts-order-service.duration_mean    175     1706151060     1706152500     0.009097     0.433339     0.484213
19      trace.to_ts-auth-service.duration_mean     50     1706151060     1706152500     0.111009     0.324009     0.370572, (220, 7))```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse df_trace and trace_agg_global if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

try:
    trace_agg_global
except NameError:
    raise RuntimeError("trace_agg_global not found in kernel. Run the global aggregation step first.")

# Ensure correct dtypes
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['ts_utc'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Target substrings (case-sensitive) and incident window (UTC)
substrs = [
    'to_ts-auth-service','to_ts-order-service','to_ts-route-service','to_ts-train-service','to_ts-travel-service',
    'from_ts-auth-service','from_ts-order-service','from_ts-route-service','from_ts-train-service','from_ts-travel-service'
]
pattern = "|".join(substrs)

start_ts = int(pd.Timestamp("2024-01-25 02:48:38", tz="UTC").timestamp())
end_ts   = int(pd.Timestamp("2024-01-25 03:18:38", tz="UTC").timestamp())

# Filter trace rows by name pattern and incident window; store in kernel variable trace_window_df
trace_window_df = df_trace[
    df_trace['trace_name'].str.contains(pattern, na=False) &
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts)
].copy()

# Prepare p95 thresholds from trace_agg_global (global, computed earlier)
p95_df = trace_agg_global[['trace_name','p95']].drop_duplicates().copy()

# Merge window rows with global p95 (inner join to keep only trace_names present in trace_agg_global)
win_with_p95 = trace_window_df.merge(p95_df, on='trace_name', how='inner')

# If no rows, produce empty summary with expected columns
if win_with_p95.empty:
    trace_anomaly_summary = pd.DataFrame(columns=[
        'trace_name','count_in_window','n_anomaly_ge_p95','earliest_anom_ts',
        'max_value_in_window','mean_value_in_window'
    ])
else:
    # Group and compute required stats
    def stats_fn(g):
        cnt = int(g['value'].count())
        n_anom = int((g['value'] >= g['p95']).sum())
        earliest = int(g.loc[g['value'] >= g['p95'], 'timestamp'].min()) if n_anom > 0 else pd.NA
        max_v = float(g['value'].max()) if cnt > 0 else pd.NA
        mean_v = float(g['value'].mean()) if cnt > 0 else pd.NA
        return pd.Series({
            'count_in_window': cnt,
            'n_anomaly_ge_p95': n_anom,
            'earliest_anom_ts': earliest,
            'max_value_in_window': max_v,
            'mean_value_in_window': mean_v
        })

    trace_anomaly_summary = win_with_p95.groupby('trace_name').apply(stats_fn).reset_index()

# Ensure types and sorting
trace_anomaly_summary['count_in_window'] = trace_anomaly_summary['count_in_window'].fillna(0).astype(int)
trace_anomaly_summary['n_anomaly_ge_p95'] = trace_anomaly_summary['n_anomaly_ge_p95'].fillna(0).astype(int)
trace_anomaly_summary['earliest_anom_ts'] = trace_anomaly_summary['earliest_anom_ts'].where(
    pd.notna(trace_anomaly_summary['earliest_anom_ts']), pd.NA
)
trace_anomaly_summary['max_value_in_window'] = trace_anomaly_summary['max_value_in_window'].astype('Float64')
trace_anomaly_summary['mean_value_in_window'] = trace_anomaly_summary['mean_value_in_window'].astype('Float64')

trace_anomaly_summary = trace_anomaly_summary.sort_values(
    by=['n_anomaly_ge_p95','max_value_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

# Limit displayed rows to top 50
trace_anomaly_summary_display = trace_anomaly_summary.head(50)

# Expose variables in kernel: trace_window_df and trace_anomaly_summary
trace_anomaly_summary_display, trace_window_df.shape

```
Out[5]:
```
```python
# Summarize the trace-window anomaly results in plain English using existing kernel variables:
# trace_anomaly_summary (DataFrame) and trace_window_df (DataFrame)

import pandas as pd

# Safety checks
if 'trace_anomaly_summary' not in globals():
    raise RuntimeError("trace_anomaly_summary not found in kernel. Run the analysis step first.")
if 'trace_window_df' not in globals():
    raise RuntimeError("trace_window_df not found in kernel. Run the filtering step first.")

df = trace_anomaly_summary.copy()
total_window_rows = int(trace_window_df.shape[0])
total_groups = int(df.shape[0])
groups_with_anom = int((df['n_anomaly_ge_p95'] > 0).sum())

# Prepare top 10 groups by anomaly count then peak value for concise display
top10 = df.sort_values(['n_anomaly_ge_p95','max_value_in_window'], ascending=[False, False]).head(10)

def fmt_num(x):
    if pd.isna(x):
        return "None"
    return f"{float(x):.6g}"

lines = []
for _, r in top10.iterrows():
    lines.append(
        f"{r['trace_name']} | count_in_window={int(r['count_in_window'])}, "
        f"n_anomaly_ge_p95={int(r['n_anomaly_ge_p95'])}, earliest_anom_ts={int(r['earliest_anom_ts']) if pd.notna(r['earliest_anom_ts']) else 'None'}, "
        f"max={fmt_num(r['max_value_in_window'])}, mean={fmt_num(r['mean_value_in_window'])}"
    )

summary = (
    "Trace-window anomaly summary (2024-01-25 02:48:38 to 2024-01-25 03:18:38 UTC):\n\n"
    f"- Total trace rows in window (filtered traces): {total_window_rows}.\n"
    f"- Number of trace_name groups analyzed: {total_groups}.\n"
    f"- Groups with >=1 value >= global p95: {groups_with_anom}.\n\n"
    "- Top 10 groups by anomaly count and peak value:\n  " + "\n  ".join(lines) + "\n\n"
    "- Key observations:\n"
    "  * Several error_rate traces (e.g. to/from order/travel/route/train auth) show n_anomaly_ge_p95 equal to the full count in-window —"
    " however their values are 0.0 (meaning the historical p95 is 0 and current values equal that baseline).\n"
    "  * Row_count traces for train/route/show large p95s globally, and some window rows exceed those thresholds.\n"
    "  * Duration-related traces also show a small number of anomalies (typically 1–2 points) with elevated max/mean values in the window.\n\n"
    "The detailed per-trace summary table is in kernel variable `trace_anomaly_summary` and the filtered rows are in `trace_window_df`."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                   trace_name  count_in_window  n_anomaly_ge_p95  earliest_anom_ts  max_value_in_window  mean_value_in_window
0        trace.to_ts-order-service.error_rate              175               175      1.706151e+09                  0.0                   0.0
1     trace.from_ts-travel-service.error_rate              150               150      1.706151e+09                  0.0                   0.0
2       trace.to_ts-travel-service.error_rate              150               150      1.706151e+09                  0.0                   0.0
3        trace.to_ts-route-service.error_rate              100               100      1.706151e+09                  0.0                   0.0
4        trace.to_ts-train-service.error_rate              100               100      1.706151e+09                  0.0                   0.0
..                                        ...              ...               ...               ...                  ...                   ...
35   trace.from_ts-auth-service.duration_mean               25                 2      1.706151e+09             0.004916               0.00253
36   trace.from_ts-route-service.duration_p95               25                 2      1.706152e+09             0.001877              0.001745
37  trace.from_ts-route-service.duration_mean               25                 2      1.706152e+09              0.00145               0.00128
38   trace.from_ts-train-service.duration_p95               25                 2      1.706152e+09             0.001446               0.00135
39  trace.from_ts-train-service.duration_mean               25                 2      1.706152e+09             0.000982              0.000929

[40 rows x 6 columns], (3400, 5))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import re

# Ensure trace_anomaly_summary is available
if 'trace_anomaly_summary' not in globals():
    raise RuntimeError("trace_anomaly_summary not found in kernel. Run prior analysis steps first.")

# Filter to rows where max_value_in_window > 0 (exclude zeros and NA)
trace_anomaly_nonzero = trace_anomaly_summary[
    trace_anomaly_summary['max_value_in_window'].notna() &
    (trace_anomaly_summary['max_value_in_window'] > 0)
].copy()

# Sort as requested
trace_anomaly_nonzero = trace_anomaly_nonzero.sort_values(
    by=['n_anomaly_ge_p95', 'max_value_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

# Select display columns and limit to top 20 for compact output
display_cols = ['trace_name','count_in_window','n_anomaly_ge_p95','earliest_anom_ts',
                'max_value_in_window','mean_value_in_window']
trace_anomaly_nonzero_display = trace_anomaly_nonzero.loc[:, display_cols].head(20)

# Extract service substrings like 'to_ts-XYZ' or 'from_ts-XYZ' from trace_name
# Pattern finds the first occurrence of to_ts-... or from_ts-... up to the next dot
service_pattern = r'(to_ts-[^\.]+|from_ts-[^\.]+)'
trace_anomaly_nonzero['service_ref'] = trace_anomaly_nonzero['trace_name'].str.extract(service_pattern, expand=False)

# Compact aggregate: distinct services and counts per service in the filtered set
service_counts = (
    trace_anomaly_nonzero['service_ref'].fillna('(unknown)')
    .value_counts()
    .reset_index()
)
service_counts.columns = ['service_ref', 'count']
service_counts = service_counts.sort_values('count', ascending=False).reset_index(drop=True)

# Also produce a compact list of distinct services
distinct_services = pd.DataFrame({'distinct_services': sorted(trace_anomaly_nonzero['service_ref'].dropna().unique())})

# Display (variable names only, compact)
trace_anomaly_nonzero_display, service_counts, distinct_services

```
Out[6]:
```
```python
# Compose a plain-English summary from existing kernel variables:
# trace_anomaly_nonzero, service_counts, distinct_services

import pandas as pd

# Safety checks
for var in ('trace_anomaly_nonzero', 'service_counts', 'distinct_services'):
    if var not in globals():
        raise RuntimeError(f"{var} not found in kernel. Run prior steps first.")

df = trace_anomaly_nonzero.copy()
svc_counts = service_counts.copy()
distinct = distinct_services.copy()

# Summary stats
n_groups = int(df.shape[0])
top5 = df.sort_values(['n_anomaly_ge_p95','max_value_in_window'], ascending=[False, False]).head(5)

def fmt_num(x):
    return "None" if pd.isna(x) else f"{float(x):.6g}"

top5_lines = []
for _, r in top5.iterrows():
    top5_lines.append(
        f"{r['trace_name']} — count_in_window={int(r['count_in_window'])}, "
        f"n_anomaly_ge_p95={int(r['n_anomaly_ge_p95'])}, earliest_anom_ts={int(r['earliest_anom_ts']) if pd.notna(r['earliest_anom_ts']) else 'None'}, "
        f"max={fmt_num(r['max_value_in_window'])}, mean={fmt_num(r['mean_value_in_window'])}"
    )

# Service counts formatting
svc_lines = [f"{row['service_ref']}({int(row['count'])})" for _, row in svc_counts.iterrows()]

summary = (
    f"Compact summary of trace groups with non-zero max in the incident window:\n\n"
    f"- Number of trace_name groups with max_value_in_window > 0: {n_groups}.\n\n"
    f"- Top 5 groups (by anomaly count then peak value):\n  " + "\n  ".join(top5_lines) + "\n\n"
    f"- Distinct services referenced (extracted from trace_name) and counts in this filtered set:\n  " + ", ".join(svc_lines) + "\n\n"
    f"- Notable observations: row_count and duration traces for order and travel services dominate the anomalies "
    f"(e.g., to_ts-order-service.row_count had {int(top5.iloc[0]['n_anomaly_ge_p95'])} anomalous points with max={fmt_num(top5.iloc[0]['max_value_in_window'])}; "
    f"from/to travel row_count and duration traces also show high counts and peaks)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  count_in_window  n_anomaly_ge_p95  earliest_anom_ts  max_value_in_window  mean_value_in_window
0          trace.to_ts-order-service.row_count              175                 9      1.706151e+09                526.0             79.211429
1       trace.to_ts-order-service.duration_p95              175                 9      1.706152e+09             0.569358              0.081397
2      trace.to_ts-order-service.duration_mean              175                 9      1.706152e+09             0.536766               0.07417
3       trace.from_ts-travel-service.row_count              150                 8      1.706151e+09               1795.0            362.946667
4         trace.to_ts-travel-service.row_count              150                 8      1.706151e+09               1795.0            261.806667
5      trace.to_ts-travel-service.duration_p95              150                 8      1.706152e+09             16.49272              2.394112
6     trace.to_ts-travel-service.duration_mean              150                 8      1.706152e+09            14.574813              1.765178
7    trace.from_ts-travel-service.duration_p95              150                 8      1.706152e+09             1.647991              0.266691
8   trace.from_ts-travel-service.duration_mean              150                 8      1.706152e+09             1.299627              0.139743
9          trace.to_ts-train-service.row_count              100                 5      1.706151e+09               2876.0                704.45
10         trace.to_ts-route-service.row_count              100                 5      1.706151e+09               2318.0                685.47
11      trace.to_ts-route-service.duration_p95              100                 5      1.706152e+09             0.017286              0.003958
12     trace.to_ts-route-service.duration_mean              100                 5      1.706152e+09             0.006722              0.003066
13      trace.to_ts-train-service.duration_p95              100                 5      1.706152e+09             0.006406              0.003713
14     trace.to_ts-train-service.duration_mean              100                 5      1.706152e+09             0.004019              0.002966
15       trace.from_ts-order-service.row_count               50                 3      1.706151e+09                526.0                 200.4
16          trace.to_ts-auth-service.row_count               50                 3      1.706151e+09                378.0                 185.1
17       trace.to_ts-auth-service.duration_p95               50                 3      1.706151e+09             2.181191              0.359327
18    trace.from_ts-order-service.duration_p95               50                 3      1.706152e+09             0.553961              0.247203
19   trace.from_ts-order-service.duration_mean               50                 3      1.706152e+09             0.521208              0.232602,               service_ref  count
0     to_ts-order-service      3
1  from_ts-travel-service      3
2    to_ts-travel-service      3
3     to_ts-train-service      3
4     to_ts-route-service      3
5   from_ts-order-service      3
6      to_ts-auth-service      3
7   from_ts-train-service      3
8   from_ts-route-service      3
9    from_ts-auth-service      3,         distinct_services
0    from_ts-auth-service
1   from_ts-order-service
2   from_ts-route-service
3   from_ts-train-service
4  from_ts-travel-service
5      to_ts-auth-service
6     to_ts-order-service
7     to_ts-route-service
8     to_ts-train-service
9    to_ts-travel-service)```
```

In [ ]:
# In[7]:

import pandas as pd

# Ensure required kernel variables exist
if 'metric_window_df' not in globals():
    raise RuntimeError("metric_window_df not found in kernel. Run prior steps to create it.")
if 'metric_agg_global' not in globals():
    raise RuntimeError("metric_agg_global not found in kernel. Run prior aggregation step first.")

#  Filter incident-window metrics for ts-travel-service mem and save to kernel variable
ts_travel_mem_window = metric_window_df[
    (metric_window_df['cmdb_id'] == 'ts-travel-service') &
    (metric_window_df['kpi_name'] == 'mem')
].copy()

# Ensure correct dtypes
ts_travel_mem_window['timestamp'] = ts_travel_mem_window['timestamp'].astype(int)
ts_travel_mem_window['value'] = pd.to_numeric(ts_travel_mem_window['value'], errors='coerce')

# 1) Aggregate summary
if not ts_travel_mem_window.empty:
    aggregate_summary = pd.DataFrame([{
        'count_in_window': int(ts_travel_mem_window['value'].count()),
        'min_timestamp': int(ts_travel_mem_window['timestamp'].min()),
        'max_timestamp': int(ts_travel_mem_window['timestamp'].max()),
        'min_value': float(ts_travel_mem_window['value'].min()),
        'max_value': float(ts_travel_mem_window['value'].max()),
        'mean_value': float(ts_travel_mem_window['value'].mean())
    }])
else:
    aggregate_summary = pd.DataFrame([{
        'count_in_window': 0,
        'min_timestamp': pd.NA,
        'max_timestamp': pd.NA,
        'min_value': pd.NA,
        'max_value': pd.NA,
        'mean_value': pd.NA
    }])

# 2) Earliest timestamps in-window where value >= global p95 for ts-travel-service mem
p95_rows = metric_agg_global[
    (metric_agg_global['cmdb_id'] == 'ts-travel-service') &
    (metric_agg_global['kpi_name'] == 'mem')
]

if not p95_rows.empty and not ts_travel_mem_window.empty:
    p95_val = float(p95_rows.iloc[0]['p95'])
    anoms = ts_travel_mem_window[ts_travel_mem_window['value'] >= p95_val].sort_values('timestamp')
    earliest_anoms = anoms[['timestamp', 'value']].head(10).reset_index(drop=True)
else:
    earliest_anoms = pd.DataFrame(columns=['timestamp', 'value'])

# 3) Sample up to 20 rows sorted by timestamp ascending (timestamp, value)
sample_rows = ts_travel_mem_window[['timestamp', 'value']].sort_values('timestamp').head(20).reset_index(drop=True)

# Expose variables and display compact outputs
aggregate_summary, earliest_anoms, sample_rows, ts_travel_mem_window.shape

```
Out[7]:
```
```python
# Summarize findings for ts-travel-service mem within the incident window using existing kernel variables:
# aggregate_summary, earliest_anoms, sample_rows, ts_travel_mem_window

import pandas as pd

# Safety checks
for var in ('aggregate_summary', 'earliest_anoms', 'sample_rows', 'ts_travel_mem_window'):
    if var not in globals():
        raise RuntimeError(f"{var} not found in kernel. Run the prior step to create it.")

agg = aggregate_summary.loc[0]
count_in_window = int(agg['count_in_window'])
min_ts = int(agg['min_timestamp']) if pd.notna(agg['min_timestamp']) else None
max_ts = int(agg['max_timestamp']) if pd.notna(agg['max_timestamp']) else None
min_val = int(agg['min_value']) if pd.notna(agg['min_value']) else None
max_val = int(agg['max_value']) if pd.notna(agg['max_value']) else None
mean_val = float(agg['mean_value']) if pd.notna(agg['mean_value']) else None

# Earliest anomaly rows (value >= global p95) - list up to 10
anom_rows = earliest_anoms.copy()
anom_list = []
for _, r in anom_rows.iterrows():
    anom_list.append(f"timestamp={int(r['timestamp'])}, value={int(r['value'])}")

# Sample first/last rows from the provided sample (up to 20)
sample = sample_rows.copy()
first_rows = sample.head(3)
last_rows = sample.tail(3)
def rows_to_str(df):
    return "; ".join([f"(ts={int(r['timestamp'])}, val={int(r['value'])})" for _, r in df.iterrows()])

summary = (
    f"ts-travel-service / mem (incident window):\n\n"
    f"1) Aggregate summary:\n"
    f"   - count_in_window: {count_in_window}\n"
    f"   - min_timestamp: {min_ts}\n"
    f"   - max_timestamp: {max_ts}\n"
    f"   - min(value): {min_val}\n"
    f"   - max(value): {max_val}\n"
    f"   - mean(value): {mean_val:.6g}\n\n"
    f"2) Earliest in-window timestamps where value >= global p95 for ts-travel-service mem:\n"
    f"   - " + (", ".join(anom_list) if anom_list else "None\n") + "\n"
    f"3) Sample of rows (sorted by timestamp, up to 20 shown in kernel variable `ts_travel_mem_window`):\n"
    f"   - first 3: {rows_to_str(first_rows)}\n"
    f"   - last 3:  {rows_to_str(last_rows)}\n\n"
    f"Notes: The full filtered rows are saved in kernel variable `ts_travel_mem_window` (shape: {ts_travel_mem_window.shape})."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   count_in_window  min_timestamp  max_timestamp    min_value     max_value    mean_value
0               25     1706151060     1706152500  287042560.0  2.932712e+08  2.886938e+08,     timestamp         value
0  1706152440  2.919164e+08
1  1706152500  2.932712e+08,      timestamp         value
0   1706151060  2.885662e+08
1   1706151120  2.870426e+08
2   1706151180  2.870783e+08
3   1706151240  2.871189e+08
4   1706151300  2.871020e+08
5   1706151360  2.870921e+08
6   1706151420  2.871108e+08
7   1706151480  2.871201e+08
8   1706151540  2.871386e+08
9   1706151600  2.871504e+08
10  1706151660  2.872338e+08
11  1706151720  2.872305e+08
12  1706151780  2.873309e+08
13  1706151840  2.876035e+08
14  1706151900  2.884100e+08
15  1706151960  2.899985e+08
16  1706152020  2.901967e+08
17  1706152080  2.902286e+08
18  1706152140  2.902512e+08
19  1706152200  2.902615e+08, (25, 5))```
```

In [ ]:
# In[8]:

import pandas as pd

# Reuse existing df_log and df_error if available, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# Ensure dtypes
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

df_error['timestamp'] = df_error['timestamp'].astype(int)
df_error['message'] = df_error['message'].astype(str)

# Incident window (UTC) as unix seconds
start_ts = int(pd.Timestamp("2024-01-25 02:48:38", tz="UTC").timestamp())
end_ts   = int(pd.Timestamp("2024-01-25 03:18:38", tz="UTC").timestamp())

# Filter log.csv for ts-travel-service in the window; save filtered rows
log_travel_window = df_log[
    (df_log['cmdb_id'] == 'ts-travel-service') &
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts)
].copy()

# Aggregate summary
if not log_travel_window.empty:
    total_rows = int(len(log_travel_window))
    min_ts = int(log_travel_window['timestamp'].min())
    max_ts = int(log_travel_window['timestamp'].max())
    sum_error_count = float(log_travel_window.loc[log_travel_window['log_name']=='log.error_count', 'value'].sum())
    # sum row_count if present
    sum_row_count = float(log_travel_window.loc[log_travel_window['log_name']=='log.row_count', 'value'].sum())
else:
    total_rows = 0
    min_ts = pd.NA
    max_ts = pd.NA
    sum_error_count = 0.0
    sum_row_count = 0.0

log_travel_summary = pd.DataFrame([{
    'total_rows_in_window': total_rows,
    'min_timestamp': min_ts,
    'max_timestamp': max_ts,
    'sum_log.error_count': sum_error_count,
    'sum_log.row_count': sum_row_count
}])

# Earliest timestamps where log.error_count > 0 (timestamp and value), limit 20
err_rows = log_travel_window[log_travel_window['log_name']=='log.error_count'].copy()
err_rows_pos = err_rows[err_rows['value'] > 0].sort_values('timestamp').head(20)
earliest_log_errors = err_rows_pos[['timestamp','value']].reset_index(drop=True)

# Sample up to 20 rows (timestamp, log_name, value) sorted by timestamp ascending
log_travel_sample = log_travel_window[['timestamp','log_name','value']].sort_values('timestamp').head(20).reset_index(drop=True)

# Now filter error_logs.csv for ts-travel-service in the same window
error_travel_filtered = df_error[
    (df_error['cmdb_id'] == 'ts-travel-service') &
    (df_error['timestamp'] >= start_ts) &
    (df_error['timestamp'] <= end_ts)
].copy()

error_travel_count = int(len(error_travel_filtered))

# Top up to 20 distinct message first-100-chars with counts
if not error_travel_filtered.empty:
    error_travel_filtered['message_short'] = error_travel_filtered['message'].str.slice(0,100)
    error_travel_msgs = error_travel_filtered['message_short'].value_counts().reset_index()
    error_travel_msgs.columns = ['message_short_100', 'count']
    error_travel_window = error_travel_msgs.head(20).reset_index(drop=True)
else:
    error_travel_window = pd.DataFrame(columns=['message_short_100','count'])

# Display compact outputs (variables saved in kernel as requested)
log_travel_summary, earliest_log_errors, log_travel_sample, log_travel_window.shape, error_travel_count, error_travel_window

```
Out[8]:
```
```python
# Compose a plain-English summary using existing kernel variables:
# log_travel_summary, earliest_log_errors, log_travel_sample, error_travel_count, error_travel_window

import pandas as pd

# Safety checks
for var in ('log_travel_summary', 'earliest_log_errors', 'log_travel_sample', 'error_travel_count', 'error_travel_window'):
    if var not in globals():
        raise RuntimeError(f"{var} not found in kernel. Run the prior steps first.")

# Extract aggregates
ls = log_travel_summary.loc[0]
total_rows = int(ls['total_rows_in_window'])
min_ts = int(ls['min_timestamp'])
max_ts = int(ls['max_timestamp'])
sum_error_count = float(ls['sum_log.error_count'])
sum_row_count = float(ls['sum_log.row_count'])

# Earliest error_count>0 rows
earliest = earliest_log_errors.copy()
if earliest.empty:
    earliest_str = "None (no log.error_count > 0 in the window)"
else:
    earliest_str = "; ".join([f"ts={int(r['timestamp'])}, value={float(r['value'])}" for _, r in earliest.iterrows()])

# Sample rows (first up to 20 sorted by timestamp)
sample = log_travel_sample.copy()
def fmt_sample(df, n=3):
    if df.empty:
        return "(no sample rows)"
    parts = []
    for _, r in df.head(n).iterrows():
        parts.append(f"(ts={int(r['timestamp'])}, name={r['log_name']}, val={int(r['value']) if pd.notna(r['value']) else 'NA'})")
    return " | ".join(parts)

first3 = fmt_sample(sample.head(3), 3)
last3 = fmt_sample(sample.tail(3), 3)

# Error logs aggregate
err_count = int(error_travel_count)
if err_count == 0:
    err_msgs_summary = "No error_log rows for ts-travel-service in the window."
else:
    # summarize up to 20 distinct message first-100-chars
    msgs = error_travel_window.copy()
    lines = [f"\"{row['message_short_100']}\"({int(row['count'])})" for _, row in msgs.iterrows()]
    err_msgs_summary = "; ".join(lines)

summary = (
    "ts-travel-service logs (incident window 2024-01-25 02:48:38 to 2024-01-25 03:18:38 UTC):\n\n"
    f"- Total rows in window: {total_rows}\n"
    f"- Min timestamp: {min_ts}\n"
    f"- Max timestamp: {max_ts}\n"
    f"- Sum(log.error_count) in-window: {sum_error_count}\n"
    f"- Sum(log.row_count) in-window: {sum_row_count}\n\n"
    f"- Earliest timestamps where log.error_count > 0: {earliest_str}\n\n"
    f"- Sample rows (by timestamp asc):\n"
    f"  - first 3: {first3}\n"
    f"  - last  3: {last3}\n\n"
    f"- error_logs for ts-travel-service in window: count={err_count}. {err_msgs_summary}\n\n"
    "Note: Filtered log rows are saved in kernel variable `log_travel_window` and the aggregate in `log_travel_summary`; "
    "filtered error logs are in `error_travel_window` (or empty if none)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows_in_window  min_timestamp  max_timestamp  sum_log.error_count  sum_log.row_count
0                    50     1706151060     1706152500                  0.0            37588.0, Empty DataFrame
Columns: [timestamp, value]
Index: [],      timestamp         log_name  value
0   1706151060  log.error_count      0
1   1706151060    log.row_count    361
2   1706151120  log.error_count      0
3   1706151120    log.row_count   1866
4   1706151180  log.error_count      0
5   1706151180    log.row_count   2045
6   1706151240  log.error_count      0
7   1706151240    log.row_count   1347
8   1706151300  log.error_count      0
9   1706151300    log.row_count   1526
10  1706151360  log.error_count      0
11  1706151360    log.row_count   1533
12  1706151420  log.error_count      0
13  1706151420    log.row_count   1619
14  1706151480  log.error_count      0
15  1706151480    log.row_count   1854
16  1706151540  log.error_count      0
17  1706151540    log.row_count   1854
18  1706151600  log.error_count      0
19  1706151600    log.row_count   1805, (50, 5), 0, Empty DataFrame
Columns: [message_short_100, count]
Index: [])```
```